# Sprawdzanie współrzędnych miejsc w Mountained there

Notatnik porównuje punkty z aplikacji z OpenStreetMap:

- **zima**: szuka najbliższej trasy zjazdowej (`piste:type=downhill`) w OSM; jeśli jest dalej niż 2 km od pinezki, miejsce trafia na listę podejrzanych,
- **lato**: wyszukuje nazwę miejsca w Nominatim i porównuje z pinezką; podejrzane są różnice powyżej 20 km.

Uruchom wszystkie komórki (Runtime → Run all). Całość trwa ok. 15–25 minut, bo publiczne serwery OSM wymagają przerw między zapytaniami. Na końcu pobierze się plik `podejrzane.csv`: wyślij go w czacie, a poprawię współrzędne w aplikacji.

In [ ]:
import json, math, time, re, urllib.request, urllib.parse
import pandas as pd

SRC = "https://raw.githubusercontent.com/BartoszKrop/Skied-Hiked-there/main/index.html"
html = urllib.request.urlopen(SRC).read().decode("utf-8")
i = html.index("const DATA = ") + len("const DATA = ")
DATA, _ = json.JSONDecoder().raw_decode(html, i)
resorts = [r for r in DATA["resorts"] if r.get("x") is not None]
hubs = [h for h in DATA["hubs"] if h.get("x") is not None]
print(len(resorts), "ośrodków,", len(hubs), "miejsc letnich")

UA = {"User-Agent": "MountainedThere-coordcheck/1.0 (github.com/BartoszKrop/Skied-Hiked-there)"}
def km(a_lat, a_lon, b_lat, b_lon):
    p = math.pi / 180
    d = 0.5 - math.cos((b_lat - a_lat) * p) / 2 + math.cos(a_lat * p) * math.cos(b_lat * p) * (1 - math.cos((b_lon - a_lon) * p)) / 2
    return 12742 * math.asin(math.sqrt(d))
def get(url, data=None, tries=4):
    for t in range(tries):
        try:
            req = urllib.request.Request(url, data=data, headers=UA)
            return json.loads(urllib.request.urlopen(req, timeout=90).read())
        except Exception as e:
            time.sleep(5 * (t + 1))
    return None

## Zima: najbliższa trasa zjazdowa w OSM

In [ ]:
OVERPASS = "https://overpass-api.de/api/interpreter"
rows = []
for n, r in enumerate(resorts):
    lat, lon = r["y"], r["x"]
    q = f'[out:json][timeout:60];way["piste:type"="downhill"](around:15000,{lat},{lon});out center;'
    res = get(OVERPASS, urllib.parse.urlencode({"data": q}).encode())
    pts = [(e["center"]["lat"], e["center"]["lon"]) for e in (res or {}).get("elements", []) if "center" in e]
    if pts:
        d = sorted((km(lat, lon, a, b), a, b) for a, b in pts)
        near = [p for p in d if p[0] <= d[0][0] + 4]
        clat = sum(p[1] for p in near) / len(near); clon = sum(p[2] for p in near) / len(near)
        rows.append({"tryb": "zima", "id": r["id"], "nazwa": r["n"], "kraj": r["c"], "lat": lat, "lon": lon, "odl_km": round(d[0][0], 2), "propozycja_lat": round(clat, 4), "propozycja_lon": round(clon, 4), "uwagi": f"{len(pts)} tras w promieniu 15 km"})
    else:
        rows.append({"tryb": "zima", "id": r["id"], "nazwa": r["n"], "kraj": r["c"], "lat": lat, "lon": lon, "odl_km": None, "propozycja_lat": None, "propozycja_lon": None, "uwagi": "brak tras w OSM w promieniu 15 km" if res is not None else "błąd zapytania"})
    if n % 25 == 0: print(n, "/", len(resorts))
    time.sleep(1.2)
zima = pd.DataFrame(rows)
zima_podejrzane = zima[(zima.odl_km.isna()) | (zima.odl_km > 2)].sort_values("odl_km", ascending=False, na_position="first")
print(len(zima_podejrzane), "podejrzanych ośrodków")
zima_podejrzane

## Lato: porównanie z wyszukiwarką Nominatim

In [ ]:
CN = {}
rows = []
for n, h in enumerate(hubs):
    lat, lon = h["y"], h["x"]
    name = re.sub(r"\s*\(.*?\)", "", h["n"]).split(" / ")[0].split(" – ")[0].strip()
    cc = (h.get("c") or "").split(",")[0].lower()
    url = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode({"q": name, "format": "json", "limit": 3, **({"countrycodes": cc} if len(cc) == 2 else {})})
    res = get(url) or []
    if res:
        best = min(res, key=lambda e: km(lat, lon, float(e["lat"]), float(e["lon"])))
        top = res[0]
        d_top = km(lat, lon, float(top["lat"]), float(top["lon"]))
        d_best = km(lat, lon, float(best["lat"]), float(best["lon"]))
        rows.append({"tryb": "lato", "id": h["id"], "nazwa": h["n"], "kraj": h.get("c"), "lat": lat, "lon": lon, "odl_km": round(d_best, 1), "propozycja_lat": round(float(top["lat"]), 4), "propozycja_lon": round(float(top["lon"]), 4), "uwagi": f'szukano: {name} · OSM: {top.get("display_name", "")[:80]} ({round(d_top, 1)} km)'})
    else:
        rows.append({"tryb": "lato", "id": h["id"], "nazwa": h["n"], "kraj": h.get("c"), "lat": lat, "lon": lon, "odl_km": None, "propozycja_lat": None, "propozycja_lon": None, "uwagi": f"nie znaleziono: {name}"})
    if n % 25 == 0: print(n, "/", len(hubs))
    time.sleep(1.1)
lato = pd.DataFrame(rows)
lato_podejrzane = lato[(lato.odl_km.isna()) | (lato.odl_km > 20)].sort_values("odl_km", ascending=False, na_position="first")
print(len(lato_podejrzane), "podejrzanych miejsc letnich")
lato_podejrzane

## Wynik do pobrania
Wyślij `podejrzane.csv` w czacie. Wiersze „nie znaleziono” nie muszą oznaczać błędu: często to nazwa regionu, którą Nominatim zna pod inną nazwą.

In [ ]:
out = pd.concat([zima_podejrzane, lato_podejrzane])
out.to_csv("podejrzane.csv", index=False)
try:
    from google.colab import files
    files.download("podejrzane.csv")
except Exception:
    print("Zapisano podejrzane.csv")
out